# предобработка и создание эмбеддиингов от различных моделей для последующей оценки эффективности

In [20]:
import json
import os
import random
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/home/decrich/PycharmProjects/PythonProject/DATA")
IMAGES_DIR = DATA_DIR / "val2017"
ANNOTATIONS_PATH = DATA_DIR / "captions_val2017.json"
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    coco_data = json.load(f)

In [23]:
img_id = {}
rows = []

for img in coco_data["images"]:
    img_id[img["id"]] = img["file_name"]
for ann in coco_data["annotations"]:
    image_id = ann["image_id"]
    caption = ann["caption"]
    file_name = img_id.get(image_id)
    rows.append({
        "image_id": image_id,
        "file_name": file_name,
        "caption_en": caption,
    })
df = pd.DataFrame(rows)

## Проверка дубликатов и качества данных

In [24]:
empty = df["caption_en"].isna() | (df["caption_en"].str.strip() == "")
print(empty.sum())

duplicate = df.duplicated(subset=["image_id", "caption_en"])
print(duplicate.sum())

miss = []
for f in df["file_name"].unique():
    if not (IMAGES_DIR / f).exists():
        miss.append(f)
print(len(miss))


0
4
0


## убираем пустые подписи и дубликаты


In [25]:
captions_df = df[~empty]
captions_df = captions_df.drop_duplicates(subset=["image_id", "caption_en"])
captions_df = captions_df[~captions_df["file_name"].isin(miss)]


In [27]:

img = captions_df["image_id"].unique().tolist()
print("осталось элементов", len(img))

if len(img) > 5000:
    sampl_img = random.sample(img, 5000)
else:
    sampl_img = img

df = captions_df[captions_df["image_id"].isin(sampl_img)].reset_index(drop=True)
df

осталось элементов 5000


,image_id,file_name,caption_en
0,179765,000000179765.jpg,A black Honda motorcycle parked in front of a ...
1,179765,000000179765.jpg,A Honda motorcycle parked in a grass driveway
2,190236,000000190236.jpg,An office cubicle with four different types of...
3,331352,000000331352.jpg,A small closed toilet in a cramped space.
4,517069,000000517069.jpg,Two women waiting at a bench next to a street.
...,...,...,...
25005,9590,000000009590.jpg,A group of men sipping drinks and talking at a...
25006,84664,000000084664.jpg,"A plate of food with some eggs, potatoes, brea..."
25007,331569,000000331569.jpg,The strawberries was sitting beside the tall g...
25008,231237,000000231237.jpg,A bunch of small red flowers in a barnacle enc...


## Колонка с русским переводом

Добавляем колонку чтобы круто выводить в веб интерефейсе найденные изображения с описанием на русском


In [28]:
df["caption_ru"] = None

df = df[["image_id", "file_name", "caption_en", "caption_ru"]]

CSV_PATH = DATA_DIR / "captions.csv"
df.to_csv(CSV_PATH, index=False)
df.head()


,image_id,file_name,caption_en,caption_ru
0,179765,000000179765.jpg,A black Honda motorcycle parked in front of a ...,None
1,179765,000000179765.jpg,A Honda motorcycle parked in a grass driveway,None
2,190236,000000190236.jpg,An office cubicle with four different types of...,None
3,331352,000000331352.jpg,A small closed toilet in a cramped space.,None
4,517069,000000517069.jpg,Two women waiting at a bench next to a street.,None


## 2. Инициализация моделей

В качестве baseline мы решили взять модель CLIP, она не была дообучена под наш датасет что и требует ТЗ, так же эта модель хранит эмбеддинги текста и картинок в одном пространстве рядом

Так же мы решили взять модель BLIP-ITM мы думаем что она покажет лучший результат, ведь было дообучена под эту задачу, но на других ихображенях,
Сравниваем две модели:



In [29]:
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from transformers import BlipProcessor, BlipForImageTextRetrieval

device = "cpu"

In [31]:
MODELS_DIR = Path("../MODELS")
CLIP_PATH = MODELS_DIR / "clip-vit-large-patch14"
BLIP_PATH = MODELS_DIR / "blip-itm-large-flickr"

clip = CLIPModel.from_pretrained(CLIP_PATH)
clip_processor = CLIPProcessor.from_pretrained(CLIP_PATH)
clip.to(device)
clip.eval()

blip = BlipForImageTextRetrieval.from_pretrained(BLIP_PATH)
blip_processor = BlipProcessor.from_pretrained(BLIP_PATH)
blip.to(device)
blip.eval()


Loading weights: 100%|██████████| 616/616 [00:00<00:00, 8129.06it/s]


BlipForImageTextRetrieval(
  (vision_model): BlipVisionModel(
    (embeddings): BlipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): BlipEncoder(
      (layers): ModuleList(
        (0-23): 24 x BlipEncoderLayer(
          (self_attn): BlipAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (projection): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): BlipMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (layer_norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        )
      )
    )
    (post_layernorm):

In [ ]:
def clip_image_emb(path):
    image = Image.open(path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = clip.get_image_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy()


def clip_text_emb(text):
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        emb = clip.get_text_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy()

In [ ]:
def blip_image_emb(path):
    image = Image.open(path).convert("RGB")
    inputs = blip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = blip.vision_model(pixel_values=inputs["pixel_values"])[0][:, 0, :]
        emb = blip.vision_proj(out)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy()


def blip_text_emb(text):
    inputs = blip_processor(text=text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = blip.text_encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )[0][:, 0, :]
        emb = blip.text_proj(out)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy()

## Расчет эмбеддингов

Эмбединги будем считать на ноутбуке с видекартой 4060, сохраним в в файлы

In [ ]:
from tqdm import tqdm
import numpy as np
import pickle

STEP = 200
OUT_DIR = Path("embeddings_output")
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
images = df.drop_duplicates(subset=["image_id"])[["image_id", "file_name"]].reset_index(drop=True)
clip_img = []
blip_img = []
img_ids = []

for i, row in tqdm(images.iterrows(), total=len(images)):
    path = IMAGES_DIR / row["file_name"]
    try:
        c = clip_image_emb(path)
        b = blip_image_emb(path)
    except Exception as e:
        print("пропустили картинку", row["file_name"], e)
        continue

    clip_img.append(c)
    blip_img.append(b)
    img_ids.append(row["image_id"])

    if (i + 1) % STEP == 0:
        with open(OUT_DIR / "checkpoint_images.pkl", "wb") as f:
            pickle.dump({"clip": clip_img, "blip": blip_img, "ids": img_ids}, f)

print(len(img_ids))

In [ ]:
clip_txt = []
blip_txt = []
txt_ids = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    text = row["caption_en"]
    try:
        c = clip_text_emb(text)
        b = blip_text_emb(text)
    except Exception as e:
        print("пропустили подпись", text, e)
        continue

    clip_txt.append(c)
    blip_txt.append(b)
    txt_ids.append(row["image_id"])

    if (i + 1) % STEP == 0:
        with open(OUT_DIR / "checkpoint_texts.pkl", "wb") as f:
            pickle.dump({"clip": clip_txt, "blip": blip_txt, "ids": txt_ids}, f)
        print("чекпоинт на", i + 1)

print("посчитали подписей", len(txt_ids))

## Сохраняем

Списки складываем в numpy массивы и сохраняем.

In [ ]:
clip_img = np.array(clip_img)
blip_img = np.array(blip_img)
clip_txt = np.array(clip_txt)
blip_txt = np.array(blip_txt)
np.save(OUT_DIR / "clip_image_embeddings.npy", clip_img)
np.save(OUT_DIR / "blip_image_embeddings.npy", blip_img)
np.save(OUT_DIR / "clip_text_embeddings.npy", clip_txt)
np.save(OUT_DIR / "blip_text_embeddings.npy", blip_txt)

pd.DataFrame({"image_id": img_ids}).to_csv(OUT_DIR / "image_ids_order.csv", index=False)
pd.DataFrame({"image_id": txt_ids}).to_csv(OUT_DIR / "caption_ids_order.csv", index=False)